# REBA — Reentreno del detector YOLO en Colab (T4)

**Objetivo:** `best.pt` mejor — detector de 3 clases (plastico/unicel/metal)
con más épocas y batch grande en GPU (el local corrió 30 épocas en CPU).

## Pasos
1. Runtime → T4 GPU.
2. Celda 1 setup, celda 2 (sube `reba_det_pack.zip`).
3. Ejecutar todo; la última celda descarga `reba_det_models.zip`
   (best.pt + best.onnx + métricas en REBA_real_val = agua real).
4. Copiar el zip al proyecto y avisar al agente.

Tiempo estimado en T4: ~40-60 min.

In [ ]:
!pip -q install ultralytics onnx onnxsim onnxruntime-gpu
import torch
print(torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
import os, zipfile, pathlib
os.makedirs('/content/reba_det', exist_ok=True)
up = files.upload()  # reba_det_pack_final.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/reba_det')
!find /content/reba_det -name '*.jpg' | wc -l

In [ ]:
# data.yaml: train = union de las 7 fuentes, val = real_val (agua real)
import yaml
data = {
  'path': '/content/reba_det',
  'train': [f'{d}/images/train' for d in [
     'REBA_synthetic','REBA_synthetic_boost','REBA_synthetic_boost2',
     'REBA_taco','REBA_ecosort','REBA_floatwaste','REBA_copypaste']],
  'val': 'REBA_real_val/images/val',
  'names': {0:'plastico',1:'unicel',2:'metal'},
}
with open('/content/reba_det/data.yaml','w') as f:
    yaml.dump(data, f)
print(open('/content/reba_det/data.yaml').read())

In [ ]:
# Entrenar yolov8s @640 — balance tamano/precision para el server CPU
from ultralytics import YOLO
m = YOLO('yolov8s.pt')
res = m.train(data='/content/reba_det/data.yaml', imgsz=640, epochs=80, batch=32,
              patience=15, optimizer='AdamW', lr0=1e-3, cos_lr=True,
              close_mosaic=10, project='/content/runs', name='det_s',
              pretrained=True, seed=0, workers=4)
print('mAP50-95:', res.box.map)

In [ ]:
# Eval final explicita sobre real_val + export ONNX
best = '/content/runs/det_s/weights/best.pt'
v = YOLO(best).val(data='/content/reba_det/data.yaml', split='val', imgsz=640)
print('real_val mAP50:', v.box.map50, 'mAP50-95:', v.box.map)
exp = YOLO(best).export(format='onnx', imgsz=640, opset=12, simplify=True)
import os, shutil, json
os.makedirs('/content/out_det', exist_ok=True)
shutil.copy(exp, '/content/out_det/best_s.onnx')
shutil.copy(best, '/content/out_det/best_s.pt')
json.dump({'map50': float(v.box.map50), 'map': float(v.box.map),
           'per_class': v.box.maps.tolist() if hasattr(v.box,'maps') else None},
          open('/content/out_det/eval.json','w'), indent=1)
print('exportado')

In [ ]:
import zipfile
from pathlib import Path as P
with zipfile.ZipFile('/content/reba_det_models.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in P('/content/out_det').glob('*'):
        z.write(f, f.name)
files.download('/content/reba_det_models.zip')